In [2]:
import os
import json
import pandas as pd
import traceback
from dotenv import load_dotenv
from groq import Groq
import re 


In [ ]:
from langchain_groq import ChatGroq

In [ ]:
from dotenv import load_dotenv
load_dotenv()

KEY = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(api_key=KEY, model_name="llama-3.1-8b-instant", temperature=0.5)

In [ ]:
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002C6F183EF90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002C6F183FCB0>, model_name='llama-3.1-8b-instant', temperature=0.5, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import PyPDF2

In [ ]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [ ]:
TEMPLATE = """
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}
"""


In [ ]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
)

In [ ]:
quiz_chain = quiz_generation_prompt | llm | StrOutputParser()

In [13]:
TEMPLATE2 = """
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity \
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities.
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [14]:
quiz_evaluation_prompt = PromptTemplate(
    input_variables=["subject", "quiz"],
    template=TEMPLATE2
)

review_chain = quiz_evaluation_prompt | llm | StrOutputParser()

In [15]:
def generate_and_review(text, number, subject, tone):
    # Step 1: Generate quiz
    quiz = quiz_chain.invoke({
        "text": text,
        "number": number,
        "subject": subject,
        "tone": tone,
        "response_json": json.dumps(RESPONSE_JSON)
    })
    
    # Step 2: Review the quiz
    review = review_chain.invoke({
        "subject": subject,
        "quiz": quiz
    })
    
    return {"quiz": quiz, "review": review}

In [17]:
file_path = r"C:\Users\jolly\mcqgen\data.txt"

with open(file_path, 'r') as file:
    TEXT = file.read()

print(TEXT)

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely availabe to generate image, audio, and videos from text prompts.

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, A

In [18]:
# Serialize the Python dictionary into a JSON-formatted string
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [39]:
NUMBER = 6
SUBJECT = "Artificial Intelligence"
TONE = "simple"

response = generate_and_review(
    text=TEXT,
    number=NUMBER,
    subject=SUBJECT,
    tone=TONE
)

# Re-parse the new response
quiz_text = response["quiz"]
json_str = quiz_text.split("### RESPONSE_JSON")[-1].strip()
json_str = re.sub(r':\s*([A-Z][^"{\[,\n}]+)"', r': "\1"', json_str)
quiz_json = json.loads(json_str)

# Display
print("="*60)
print("GENERATED MCQ QUIZ")
print("="*60)

for key, value in quiz_json.items():
    print(f"\nQ{key}: {value['mcq']}")
    print(f"   a) {value['options']['a']}")
    print(f"   b) {value['options']['b']}")
    print(f"   c) {value['options']['c']}")
    print(f"   d) {value['options']['d']}")
    print(f"   ✅ Correct Answer: ({value['correct']})")
    print("-"*60)

GENERATED MCQ QUIZ

Q1: What is the primary goal of Artificial Intelligence (AI) research?
   a) Developing robots that can perform tasks
   b) Creating machines that can think and learn like humans
   c) Improving computer graphics
   d) Enhancing software security
   ✅ Correct Answer: (b)
------------------------------------------------------------

Q2: What is an example of a high-profile application of AI?
   a) Virtual reality games
   b) Autonomous vehicles
   c) Web search engines
   d) All of the above
   ✅ Correct Answer: (d)
------------------------------------------------------------

Q3: What is Artificial General Intelligence (AGI)?
   a) AI that can complete any cognitive task
   b) AI that can only perform a limited set of tasks
   c) AI that can only understand human language
   d) AI that can only recognize images
   ✅ Correct Answer: (a)
------------------------------------------------------------

Q4: Why did AI research experience a decline in funding and interest?


#CHOICE IS YOURS 


In [40]:
quiz_table_data = []

for key, value in quiz_json.items():
    mcq = value["mcq"]
    options = " | ".join(
        [f"{option}: {option_value}" for option, option_value in value["options"].items()]
    )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

# Display as a pandas dataframe
quiz_df = pd.DataFrame(quiz_table_data)
quiz_df

,MCQ,Choices,Correct
0,What is the primary goal of Artificial Intelligence (AI) research?,a: Developing robots that can perform tasks | b: Creating machines that can think and learn like humans | c: Improving computer graphics | d: Enhancing software security,b
1,What is an example of a high-profile application of AI?,a: Virtual reality games | b: Autonomous vehicles | c: Web search engines | d: All of the above,d
2,What is Artificial General Intelligence (AGI)?,a: AI that can complete any cognitive task | b: AI that can only perform a limited set of tasks | c: AI that can only understand human language | d: AI that can only recognize images,a
3,Why did AI research experience a decline in funding and interest?,a: Due to advancements in AI techniques | b: Due to the lack of real-world applications | c: Due to the failure to meet expectations | d: Due to the loss of funding and lack of interest,c
4,What is the term for the rapid increase in speed that occurs as problems grow in size?,a: Combinatorial explosion | b: Algorithmic complexity | c: Exponential growth | d: Polynomial growth,a
5,What is a key challenge in AI research?,a: Creating machines that can think and learn like humans | b: Improving computer graphics | c: Accurate and efficient reasoning | d: Enhancing software security,c


In [41]:
quiz_df.to_csv("AI.csv", index=False)